# Track A: the news MCP wrapper on Container Apps

Put a server you own in front of the GNews MCP server, give `news-desk-agent` a tool that reaches it with the user's delegated token, and show that user A and user B get different news from the same agent.

[`news-mcp-wrapper/server.py`](news-mcp-wrapper/server.py) is a FastMCP server. It validates the Entra token Foundry sends on the user's behalf, lists only the tools the token's app roles grant, answers any other call with a denial that names the allowed categories, and forwards allowed calls to GNews with the GNews key, which never leaves the container. [08-05c-00](08-05c-00-news-mcp-per-user-access.md) compares it with the gateway approach of [08-05c-03](08-05c-03-apim-passthrough.ipynb).

**What this notebook does:**
1. Creates `rg-foundry-news-mcp-{suffix}` and deploys the registry, the pull identity, the log workspace and the Container Apps environment
2. Builds the wrapper image in the registry
3. Deploys the container app and waits for `/health`
4. Creates the `news-mcp-wrapper` connection and a `news-desk-agent` version that uses it
5. Signs in user A and user B
6. Reads the wrapper's logs to see which claims the delegated token carries
7. Runs the test matrix as both users
8. Compares each user's `tools/list`, through the agent and directly
9. Calls the agent with your Azure CLI token for Foundry instead of a blueprint token
10. Runs the per-agent variant: two agents, each with its own identity and roles
11. Revokes user B's `News.Read.Science` and records the time for 08-05c-03

## Prerequisites

1. **08-05c-01** has run: it wrote the app, user, blueprint and connection settings to `.env`.
2. **A GNews key**: register at [gnews.io](https://gnews.io) (free plan, 100 requests a day) and add `GNEWS_API_KEY=...` to `.env`. This notebook makes about seven GNews requests; denied calls make none.
3. **Azure CLI** as in 08-05c-01. You need Owner (or Contributor plus User Access Administrator) on the subscription, to create the resource group and the AcrPull role assignment.
4. **User B's pass** from 08-05c-01 is valid for 8 hours. If it has expired, re-run Step 5 of 08-05c-01 for a new one.
5. Optional: `NEWS_MCP_RESOURCE_GROUP` and `NEWS_MCP_LOCATION` in `.env` override the resource group name and region (default `swedencentral`).

## Step 0: Configuration

In [1]:
import hashlib, json, os, subprocess, tempfile, time
from datetime import datetime, timezone
from pathlib import Path

import requests
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

import news_mcp_checks as checks
import news_mcp_identity as ident
from news_mcp_identity import authz

repo_root = Path(subprocess.run('git rev-parse --show-toplevel', shell=True, capture_output=True, text=True).stdout.strip())
ENV_FILE = repo_root / '.env'
load_dotenv(ENV_FILE, override=True)

REQUIRED = ['NEWS_MCP_RESOURCE_APP_ID', 'NEWS_MCP_RESOURCE_SP_ID', 'NEWS_DESK_CLIENT_APP_ID', 'NEWS_TEST_USER_UPN',
            'NEWS_TEST_USER_ID', 'NEWS_AGENT_BLUEPRINT_CLIENT_ID', 'NEWS_CONNECTION_AUTH_TYPE',
            'NEWS_CONNECTION_API_VERSION', 'GNEWS_API_KEY']
missing = [key for key in REQUIRED if not os.environ.get(key)]
if missing:
    raise SystemExit(f"Missing in .env: {', '.join(missing)}. Run 08-05c-01 first, and add GNEWS_API_KEY.")

account = json.loads(subprocess.run('az account show -o json', shell=True, capture_output=True, text=True).stdout)
SUBSCRIPTION_ID, TENANT_ID = account['id'], account['tenantId']
SUFFIX = hashlib.sha256((SUBSCRIPTION_ID + 'v2').encode()).hexdigest()[:6]
CHAT_MODEL = os.environ.get('CHAT_MODEL', 'gpt-5.4-mini')
PROJECT_ENDPOINT = f'https://aif-core-{SUFFIX}.services.ai.azure.com/api/projects/project-admin-{SUFFIX}'

RESOURCE_APP_ID = os.environ['NEWS_MCP_RESOURCE_APP_ID']
RESOURCE_SP_ID = os.environ['NEWS_MCP_RESOURCE_SP_ID']
CLIENT_APP_ID = os.environ['NEWS_DESK_CLIENT_APP_ID']
BLUEPRINT_CLIENT_ID = os.environ['NEWS_AGENT_BLUEPRINT_CLIENT_ID']
TEST_USER_ID = os.environ['NEWS_TEST_USER_ID']
AUTH_TYPE, CONNECTION_API = os.environ['NEWS_CONNECTION_AUTH_TYPE'], os.environ['NEWS_CONNECTION_API_VERSION']

RG = os.environ.get('NEWS_MCP_RESOURCE_GROUP') or f'rg-foundry-news-mcp-{SUFFIX}'
LOCATION = os.environ.get('NEWS_MCP_LOCATION') or 'swedencentral'
APP_NAME = f'ca-news-mcp-{SUFFIX}'
LAB_DIR = Path.cwd()

cli = AzureCliCredential()
ARM, FOUNDRY = 'https://management.azure.com/.default', 'https://ai.azure.com/.default'
project = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=cli, allow_preview=True)


def graph(method, path, body=None):
    """Microsoft Graph v1.0 as the Azure CLI user."""
    r = requests.request(method, f'https://graph.microsoft.com/v1.0{path}', json=body, timeout=60,
                         headers={'Authorization': f"Bearer {cli.get_token('https://graph.microsoft.com/.default').token}"})
    if not r.ok:
        raise RuntimeError(f'{method} {path}: HTTP {r.status_code} {r.text[:600]}')
    return r.json() if r.content else None


def call(method, url, scope, body=None):
    """ARM or the Foundry data plane with a CLI token for `scope`. The caller checks the response."""
    return requests.request(method, url, data=None if body is None else json.dumps(body), timeout=120,
                            headers={'Authorization': f'Bearer {cli.get_token(scope).token}', 'Content-Type': 'application/json'})


def env_set(**values):
    """Write keys to the repo .env: replace the key's line if present, append it otherwise."""
    lines = ENV_FILE.read_text().splitlines()
    for key, value in values.items():
        os.environ[key] = value
        at = [i for i, line in enumerate(lines) if line.split('=', 1)[0].strip() == key]
        if at:
            lines[at[0]] = f'{key}={value}'
        else:
            lines.append(f'{key}={value}')
    ENV_FILE.write_text('\n'.join(lines) + '\n')


def agent_identity(name):
    """The agent identity (principal_id, client_id) of an agent's latest version."""
    r = call('GET', f'{PROJECT_ENDPOINT}/agents/{name}?api-version={ident.AGENT_API_VERSION}', FOUNDRY)
    r.raise_for_status()
    return r.json()['versions']['latest']['instance_identity']


def az(*args):
    """Run an az command; return its JSON output, or raise with the end of stderr."""
    r = subprocess.run(['az', *args, '-o', 'json'], capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(f"az {' '.join(args[:3])}: {r.stderr.strip()[-1500:]}")
    return json.loads(r.stdout) if r.stdout.strip() else None


print(f'Suffix          : {SUFFIX}')
print(f'Resource group  : {RG} ({LOCATION})')
print(f'Container app   : {APP_NAME}')
print(f'Connection auth : {AUTH_TYPE} (ARM api-version {CONNECTION_API})')

Suffix          : c2676f
Resource group  : rg-foundry-news-mcp-c2676f (swedencentral)
Container app   : ca-news-mcp-c2676f
Connection auth : AgentOnBehalfOfUser (ARM api-version 2025-04-01-preview)


## Step 1: Resource group and first deployment

[`infra/wrapper.bicep`](infra/wrapper.bicep) is deployed twice. This first pass (`deployApp=false`) creates a Log Analytics workspace, a Basic container registry with no admin user, the user-assigned identity `id-news-mcp-{suffix}` with AcrPull on the registry, and the Container Apps environment. The container app needs an image that does not exist yet, so it comes in Step 3.

The resource group carries `SecurityControl=Ignore`, as in 08-05 and 08-05b: some governed subscriptions run a policy that rewrites public network settings, and this tag is its exclusion. The GNews key reaches the deployment through a parameters file that is deleted right after, never on a command line.

In [2]:
az('group', 'create', '-n', RG, '-l', LOCATION, '--tags', 'SecurityControl=Ignore')
AUDIENCES = [RESOURCE_APP_ID, authz.app_id_uri(SUFFIX)]


def deploy(deploy_app, image=''):
    """Deploy infra/wrapper.bicep and return its outputs."""
    params = {'suffix': SUFFIX, 'tenantId': TENANT_ID, 'audiences': AUDIENCES,
              'gnewsApiKey': os.environ['GNEWS_API_KEY'], 'deployApp': deploy_app, 'image': image}
    with tempfile.NamedTemporaryFile('w', suffix='.json', delete=False) as f:
        json.dump({'$schema': 'https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#',
                   'contentVersion': '1.0.0.0', 'parameters': {k: {'value': v} for k, v in params.items()}}, f)
    try:
        result = az('deployment', 'group', 'create', '-g', RG, '-n', 'news-mcp-wrapper',
                    '--template-file', str(LAB_DIR / 'infra' / 'wrapper.bicep'), '--parameters', f'@{f.name}')
    finally:
        os.unlink(f.name)
    return {k: v['value'] for k, v in result['properties']['outputs'].items()}


t0 = time.time()
outputs = deploy(False)
REGISTRY, LOGIN_SERVER = outputs['registryName'], outputs['registryLoginServer']
print(f'Deployed in {time.time() - t0:.0f}s')
for resource in az('resource', 'list', '-g', RG, '--query', '[].{type:type, name:name}'):
    print(f"  {resource['type']:<55} {resource['name']}")

Deployed in 161s
  Microsoft.ManagedIdentity/userAssignedIdentities        id-news-mcp-c2676f
  Microsoft.OperationalInsights/workspaces                log-news-mcp-c2676f
  Microsoft.ContainerRegistry/registries                  acrnewsmcpc2676f
  Microsoft.App/managedEnvironments                       cae-news-mcp-c2676f


## Step 2: Build the image

`az acr build` sends [`news-mcp-wrapper/`](news-mcp-wrapper/) to the registry, which builds the image there; Docker is not needed on this machine. The container app references the image by digest, so every rebuild becomes a new revision.

In [3]:
t0 = time.time()
r = subprocess.run(['az', 'acr', 'build', '-r', REGISTRY, '-t', 'news-mcp-wrapper:latest', '--platform', 'linux/amd64',
                    '--no-logs', str(LAB_DIR / 'news-mcp-wrapper')], capture_output=True, text=True)
if r.returncode:
    raise RuntimeError(r.stderr[-1500:])
digest = az('acr', 'repository', 'show', '-n', REGISTRY, '--image', 'news-mcp-wrapper:latest', '--query', 'digest')
IMAGE = f'{LOGIN_SERVER}/news-mcp-wrapper@{digest}'
print(f'Built in {time.time() - t0:.0f}s: {IMAGE}')

Built in 69s: acrnewsmcpc2676f.azurecr.io/news-mcp-wrapper@sha256:8b1ef336ce77372b43618345417e7d0384592b7733c63af6cf851f517abaa4a4


## Step 3: Deploy the container app

The second pass adds `ca-news-mcp-{suffix}`: external HTTPS ingress to port 8080, one to two replicas (one always on, because a cold start would eat into the 100 seconds Foundry gives an MCP call), the GNews key as a Container Apps secret, and liveness and readiness probes on `/health`. The cell waits for `/health`, then checks that the MCP endpoint turns away a request without a token.

In [4]:
t0 = time.time()
outputs = deploy(True, IMAGE)
WRAPPER_MCP_URL = outputs['wrapperMcpUrl']
base = WRAPPER_MCP_URL.removesuffix('/mcp')
for _ in range(60):
    try:
        if requests.get(f'{base}/health', timeout=10).status_code == 200:
            break
    except requests.RequestException:
        pass
    time.sleep(5)
print(f'Deployed in {time.time() - t0:.0f}s: {WRAPPER_MCP_URL}')
print(f"GET  /health without a token: HTTP {requests.get(f'{base}/health', timeout=10).status_code}")
print(f"POST /mcp    without a token: HTTP {requests.post(WRAPPER_MCP_URL, json={}, timeout=10).status_code}")
env_set(NEWS_MCP_WRAPPER_URL=WRAPPER_MCP_URL)

Deployed in 92s: https://ca-news-mcp-c2676f.ashyriver-d58ec5f4.swedencentral.azurecontainerapps.io/mcp
GET  /health without a token: HTTP 200
POST /mcp    without a token: HTTP 401


## Step 4: The connection and the agent's tool

`news-mcp-wrapper` is the same kind of connection as `news-mcp-apim` from 08-05c-01 (same auth type, same `scopes`), pointing at the wrapper. The new `news-desk-agent` version adds one MCP tool that names the connection; the agent endpoint routes to the latest version.

In [25]:
def put_connection(name, body):
    r = call('PUT', ident.connection_url(SUBSCRIPTION_ID, SUFFIX, name, CONNECTION_API), ARM, body)
    if not r.ok:
        raise RuntimeError(f'PUT connection {name}: HTTP {r.status_code} {r.text[:600]}')
    return r.json()


connection = put_connection('news-mcp-wrapper', ident.delegated_connection_body(WRAPPER_MCP_URL, SUFFIX, AUTH_TYPE))
print(f"Connection news-mcp-wrapper: authType={connection['properties']['authType']}  scopes={connection['properties'].get('scopes')}")

agent = project.agents.create_version(agent_name=ident.AGENT_NAME, definition=PromptAgentDefinition(
    model=CHAT_MODEL, instructions=ident.AGENT_INSTRUCTIONS,
    tools=[ident.mcp_tool('news_mcp_wrapper', WRAPPER_MCP_URL, connection['id'])]))
TRACK_A_VERSION = agent.version
print(f'{agent.name} v{agent.version}: tool news_mcp_wrapper -> {WRAPPER_MCP_URL}')

Connection news-mcp-wrapper: authType=AgentOnBehalfOfUser  scopes=['api://news-mcp-c2676f/News.Read']
news-desk-agent v4: tool news_mcp_wrapper -> https://ca-news-mcp-c2676f.ashyriver-d58ec5f4.swedencentral.azurecontainerapps.io/mcp


## Sign in as user A and user B

Two device code sign-ins, as in 08-05c-01. Each cell prints the account it expects and stops if the token belongs to someone else.

| Cell | Sign in as | Account | Method |
|---|---|---|---|
| first | user A | your admin account, the one `az login` uses | your usual password and multifactor method |
| second | user B | `news-sports-desk@{your tenant's default domain}`, in `.env` as `NEWS_TEST_USER_UPN` | Temporary Access Pass, in `.env` as `NEWS_TEST_USER_TAP` |

`grep -E 'NEWS_TEST_USER_(UPN|TAP)' .env` from the repository root shows user B's account and pass.

For user B:

1. Open a private browser window. A normal window offers the account you are already signed in with, and picking it signs in user A instead.
2. Open the address the cell prints and enter its code.
3. The page names `news-desk-client-{suffix}` and the location of the machine that runs the kernel. That is expected: continue.
4. If an account picker appears, choose **Use another account** and type user B's account.
5. If Entra asks for a password, choose **Other ways to sign in**, then **Temporary Access Pass**, and paste the pass.

Each client then carries the user's token for the agent blueprint's `access_agent` scope.

In [6]:
print('User A, your admin account:')
cred_a = checks.user_credential(TENANT_ID, CLIENT_APP_ID, device_code=True)
token = cred_a.get_token(f'api://{BLUEPRINT_CLIENT_ID}/access_agent').token  # the sign-in; cred_a caches it
if checks.jwt_claims(token)['oid'] == TEST_USER_ID:
    raise RuntimeError('That sign-in was user B. Sign in as user A, your admin account, and re-run this cell.')

User A, your admin account:
To sign in, use a web browser to open the page https://login.microsoft.com/device and enter the code EY7AU5BN5 to authenticate.


In [21]:
print(f"User B, {os.environ['NEWS_TEST_USER_UPN']}, in a private window, with the Temporary Access Pass in .env (NEWS_TEST_USER_TAP):")
cred_b = checks.user_credential(TENANT_ID, CLIENT_APP_ID, device_code=True)
token = cred_b.get_token(f'api://{BLUEPRINT_CLIENT_ID}/access_agent').token  # the sign-in
if checks.jwt_claims(token)['oid'] != TEST_USER_ID:
    raise RuntimeError("That sign-in was not user B. A device code page reuses any account already signed in to the browser: open a private window, choose \"Use another account\", sign in as user B with the pass, and re-run this cell.")
print('Signed in as user B')

clients = {user: checks.agent_client(cred, PROJECT_ENDPOINT, ident.AGENT_NAME, BLUEPRINT_CLIENT_ID)
           for user, cred in (('A', cred_a), ('B', cred_b))}
ROLES = {'A': ['News.Read.All'], 'B': ['News.Read.Sports', 'News.Read.Science', 'News.Read.Health']}
print('\nClients ready for user A and user B')

User B, {user}@{tenant}.onmicrosoft.com, in a private window:
To sign in, use a web browser to open the page https://login.microsoft.com/device and enter the code F5LDJ6KGC to authenticate.
Signed in as user B

Clients ready for user A and user B


## Step 5: What the wrapper sees

User A asks for business headlines. The wrapper writes one JSON line per event to stdout, and `az containerapp logs show` returns the recent ones. The `initialize` line shows the MCP protocol version Foundry asked for; the `tools/call` line shows the claims of the token Foundry obtained on user A's behalf:

| Claim | Expected |
|---|---|
| `idtyp` | `user`: a delegated token |
| `oid` | user A's object id |
| `azp` | the client id of the agent identity that asked for the token (not the blueprint) |
| `roles` | `News.Read.All`, from the assignment in 08-05c-01 |
| `scp` | `News.Read` |
| `aud` | the client id of `news-mcp-{suffix}` (v2 token) |

In [26]:
def wrapper_events(kind=None, tail=300):
    """The wrapper's recent JSON events from the container app's console log."""
    r = subprocess.run(['az', 'containerapp', 'logs', 'show', '-n', APP_NAME, '-g', RG, '--type', 'console',
                        '--tail', str(tail), '--format', 'json'], capture_output=True, text=True)
    return checks.wrapper_events(r.stdout.splitlines(), kind)


response = clients['A'].responses.create(input=checks.prompt('business')).model_dump(warnings=False)
print(f"User A, business: {checks.classify(response, ROLES['A'], 'business')}, tools called: {checks.called_tools(response)}\n")
time.sleep(10)  # the log stream lags a few seconds

session = wrapper_events('initialize')[-1]
claims = wrapper_events('tools/call')[-1]
print(f"initialize : client={session['client']}  requested={session['requested']}  negotiated={session['negotiated']}")
print(f"MCP-Protocol-Version header on tools/call: {claims['mcp_protocol_version']}")
for key in ('idtyp', 'oid', 'azp', 'appid', 'roles', 'scp', 'aud', 'tool', 'decision'):
    if key in claims:
        print(f'  {key:<9} {claims[key]}')
print(f"\nazp is news-desk-agent's agent identity: {claims.get('azp') == agent_identity(ident.AGENT_NAME)['client_id']}")

User A, business: articles, tools called: [('get_business_headlines', {'max': 3, 'lang': 'en'})]

initialize : client=ToolServer  requested=2025-11-25  negotiated=2025-11-25
MCP-Protocol-Version header on tools/call: 2025-11-25
  idtyp     user
  oid       00000000-0000-0000-0000-000000000000
  azp       00000000-0000-0000-0000-000000000000
  roles     ['News.Read.All']
  scp       News.Read
  aud       00000000-0000-0000-0000-000000000000
  tool      get_business_headlines
  decision  allowed

azp is news-desk-agent's agent identity: True


**Observed on 2026-10-30.** Foundry's MCP client calls itself `ToolServer`, asks for protocol `2025-11-25` and sends `MCP-Protocol-Version: 2025-11-25` on each `tools/call`. For each response it opens a new session: `initialize`, a `GET /mcp` for a server-sent event stream (the stateless wrapper answers 405 and Foundry carries on), `tools/list`, then the calls. `azp` is the client id of `news-desk-agent`'s agent identity, not the blueprint's.

## Step 6: The test matrix

The same three prompts run on both tracks, from [`news_mcp_checks.py`](news_mcp_checks.py), plus user B asking for science, which user B may still read here. Each line shows the tools the agent listed, the tools it called, and the outcome:

| Outcome | Meaning |
|---|---|
| `articles` | a call for the requested category returned headlines |
| `hidden` | the server listed no tool for the category, and the agent called nothing (or called the tool anyway and Foundry refused it as not listed) |
| `denied` | a call came back with the denial text, which names the user's categories in GNews order |
| `substituted` | the agent fetched another category instead; the matrix fails |

On this track user B asking for business is `hidden`. The wrapper leaves `get_business_headlines` out of user B's `tools/list`, so the agent has no tool to call and tells user B that their access does not include business. Step 7 calls the tool directly to show the denial text. On 08-05c-03, where everyone sees GNews's own tools, the same case is `denied`.

The wrapper's log for the same calls follows: one `tools/call` line per call with the decision, and one `upstream` line per call that reached GNews. User B's business request leaves no line at all.

In [27]:
cases = (*checks.base_cases('wrapper'), checks.science_case('articles'))
records = checks.run_matrix(clients, cases, ROLES)
time.sleep(10)

events = [e for e in wrapper_events() if e['event'] in ('tools/call', 'upstream')]
calls = [i for i, e in enumerate(events) if e['event'] == 'tools/call']
made = sum(len(r['called']) for r in records)  # the hidden case makes none
who = {os.environ['NEWS_TEST_USER_ID']: 'B'}
print('\nWrapper log for the matrix:')
for e in events[calls[-made]:]:
    if e['event'] == 'tools/call':
        print(f"  tools/call  user {who.get(e['oid'], 'A')}  {e['tool']:<24} {e['decision']}")
    else:
        print(f"      upstream  {e['tool']}({', '.join(f'{k}={v}' for k, v in e['arguments'].items())})")

A | Give me three business headlines    | listed 10 | called get_business_headlines   | articles    | expected articles | PASS
B | Give me three business headlines    | listed  3 | called -                        | hidden      | expected hidden   | PASS
B | Give me three sports headlines      | listed  3 | called get_sports_headlines     | articles    | expected articles | PASS
B | Give me three science headlines     | listed  3 | called get_science_headlines    | articles    | expected articles | PASS
GNews calls so far: 3 (budget for a full run: 8)

Wrapper log for the matrix:
  tools/call  user A  get_business_headlines   allowed
      upstream  top_headlines(category=business, max=3, lang=en)
  tools/call  user B  get_sports_headlines     allowed
      upstream  top_headlines(category=sports, max=3, lang=en)
  tools/call  user B  get_science_headlines    allowed
      upstream  top_headlines(category=science, max=3, lang=en)


**Observed on 2026-10-30.** With earlier instructions that only said not to retry after a denial, user B's business request went two ways in two runs. Once the agent replied that it could not fetch business headlines with the available tools. Once it called `get_science_headlines` and answered with science headlines: the model filled the gap with a tool it had. The instructions now add "If no news tool covers the category the user asks for, say that their access does not include it and do not fetch another category instead", and the matrix fails a `substituted` outcome so a regression shows.

## Step 7: Each user's `tools/list`

The agent lists the server's tools before every response, so the matrix records already show what each user saw. The second check skips the agent: each user's own token for `api://news-mcp-{suffix}/News.Read` goes straight to the wrapper's `tools/list`. The wrapper runs stateless, so the request needs no `initialize` first.

Last, user B calls `get_business_headlines` directly with the same token. The wrapper answers with `isError: true` and the denial text, and its log shows the call denied with no `upstream` line. A client that skips `tools/list`, or holds an old list, still gets no business news.

In [37]:
for user in ('A', 'B'):
    listed = next(r['listed'] for r in records if r['user'] == user and r['listed'])
    print(f'User {user} through the agent: {len(listed)} tools  {listed}')
print()
tokens = {}
for user, cred in (('A', cred_a), ('B', cred_b)):
    tokens[user] = cred.get_token(f'{authz.app_id_uri(SUFFIX)}/{authz.SCOPE}').token
    tools = checks.list_tools_direct(WRAPPER_MCP_URL, tokens[user])
    print(f'User {user} directly       : {len(tools)} tools  {tools}')

hidden_tool = authz.tool_for('business')
result = checks.call_tool_direct(WRAPPER_MCP_URL, tokens['B'], hidden_tool, {'max': 3})
denial = result['content'][0]['text']
print(f"\nUser B calls {hidden_tool} directly: isError={result['isError']}\n  {denial}")
assert result['isError'] and denial == authz.denial_message(ROLES['B'])
time.sleep(10)  # the log stream lags a few seconds
events = [e for e in wrapper_events() if e['event'] in ('tools/call', 'upstream')]
last = max(i for i, e in enumerate(events) if e['event'] == 'tools/call')
print(f"Wrapper log: tools/call {events[last]['tool']} {events[last]['decision']}, "
      f"upstream lines after it: {sum(e['event'] == 'upstream' for e in events[last + 1:])}")

User A through the agent: 10 tools  ['get_general_headlines', 'get_world_headlines', 'get_nation_headlines', 'get_business_headlines', 'get_technology_headlines', 'get_entertainment_headlines', 'get_sports_headlines', 'get_science_headlines', 'get_health_headlines', 'search_news']
User B through the agent: 3 tools  ['get_sports_headlines', 'get_science_headlines', 'get_health_headlines']

User A directly       : 10 tools  ['get_general_headlines', 'get_world_headlines', 'get_nation_headlines', 'get_business_headlines', 'get_technology_headlines', 'get_entertainment_headlines', 'get_sports_headlines', 'get_science_headlines', 'get_health_headlines', 'search_news']
User B directly       : 3 tools  ['get_sports_headlines', 'get_science_headlines', 'get_health_headlines']

User B calls get_business_headlines directly: isError=True
  Your access covers: sports, science, health. Do not retry other categories.
Wrapper log: tools/call get_business_headlines denied, upstream lines after it: 0


## Step 8: A Foundry token

The endpoint still accepts Foundry tokens (the `AzureRbac` scheme). `get_openai_client` sends the Azure CLI token for `https://ai.azure.com`, with no `access_agent` scope. The cell sends the business prompt that way, prints what comes back, and prints whose token the wrapper received.

In [33]:
foundry_client = project.get_openai_client(agent_name=ident.AGENT_NAME)
try:
    result = foundry_client.responses.create(input=checks.prompt('business'))
    response = result.model_dump(warnings=False)
    print(f"Output items: {[item['type'] for item in response['output']]}")
    for item in response['output']:
        if item['type'] == 'mcp_call':
            print(f"  mcp_call {item['name']}: error={str(item.get('error'))[:300]}  output={str(item.get('output'))[:200]}")
    print(f'Model: {result.output_text[:400]}')
    time.sleep(10)  # the log stream lags a few seconds
    e = wrapper_events('tools/call')[-1]
    print(f"\nWrapper saw: idtyp={e['idtyp']}  user A={e['oid'] == graph('GET', '/me')['id']}  "
          f"azp is news-desk-agent's identity={e.get('azp') == agent_identity(ident.AGENT_NAME)['client_id']}  "
          f"roles={e['roles']}  {e['tool']} {e['decision']}")
except Exception as e:
    print(f'{type(e).__name__}: {str(e)[:600]}')

Output items: ['mcp_list_tools', 'mcp_call', 'message']
  mcp_call get_business_headlines: error=None  output={"information":{"realTimeArticles":{"message":"Real-time news data is only available on paid plans. Free plan has a 12-hour delay. Upgrade your plan here to remove the delay: https://gnews.io/change-p
Model: Here are three business headlines:

1. AI Hyperscalers Expected to Spend up to a Whopping $828B in 2026  
2. Inside Chery’s plan to take over Australian roads  
3. Kenya Weighs Fuel Efficiency Rules

Wrapper saw: idtyp=user  user A=True  azp is news-desk-agent's identity=True  roles=['News.Read.All']  get_business_headlines allowed


**Observed on 2026-10-30.** The call returned business headlines. The wrapper received a delegated token for user A: `idtyp=user`, user A's `oid`, `azp` the agent identity of `news-desk-agent`, `roles` `News.Read.All`. The Azure CLI token is user A's own token, so Foundry exchanged it on user A's behalf, as it does a token from the `AgentBlueprint` scheme, and the wrapper applied user A's roles. Keeping `AzureRbac` on the endpoint does not open a path around the per-user rules for a signed-in user.

## Step 9: The per-agent variant

When no test user can be created, or when entitlements belong to agents rather than people, each agent gets its own identity and its own app roles: `news-all-agent` holds `News.Read.All`, `news-sports-agent` holds `News.Read.Sports`. Their connections use the agent identity (`AgenticIdentityToken` with the audience `api://news-mcp-{suffix}`), so the token the wrapper sees is app-only: `idtyp=app`, `oid` the agent identity's principal, `azp` its client id, its roles, no user. The wrapper code does not change, and neither does the outcome: `news-sports-agent` asking for business is `hidden`, as for user B.

In [34]:
# agent: (app role, connection)
FALLBACK = {'news-all-agent': ('News.Read.All', 'news-mcp-wrapper-all'),
            'news-sports-agent': ('News.Read.Sports', 'news-mcp-wrapper-sports')}
assigned_to = f'/servicePrincipals/{RESOURCE_SP_ID}/appRoleAssignedTo'


def assign(body):
    """POST an app role assignment. A new agent identity takes a minute or two to reach Graph v1.0;
    until then the POST fails with 'Not a valid reference update'."""
    for _ in range(18):
        try:
            return graph('POST', assigned_to, body)
        except RuntimeError as e:
            if 'Not a valid reference update' not in str(e):
                raise
            time.sleep(10)
    raise RuntimeError(f"Agent identity {body['principalId']} did not reach Graph within 3 minutes")


fallback_ids = {}
for name, (role, connection_name) in FALLBACK.items():
    connection = put_connection(connection_name, ident.agentic_connection_body(WRAPPER_MCP_URL, SUFFIX))
    agent = project.agents.create_version(agent_name=name, definition=PromptAgentDefinition(
        model=CHAT_MODEL, instructions=ident.AGENT_INSTRUCTIONS,
        tools=[ident.mcp_tool('news_mcp_wrapper', WRAPPER_MCP_URL, connection['id'])]))
    identity = agent_identity(name)
    fallback_ids[identity['principal_id']] = name
    body = ident.role_assignment_body(SUFFIX, identity['principal_id'], RESOURCE_SP_ID, role)
    if body['appRoleId'] not in {a['appRoleId'] for a in graph('GET', assigned_to)['value'] if a['principalId'] == identity['principal_id']}:
        assign(body)
    print(f"{name} v{agent.version}: connection {connection['name']} ({connection['properties']['authType']}), role {role}")

news-all-agent v1: connection news-mcp-wrapper-all (AgenticIdentityToken), role News.Read.All
news-sports-agent v1: connection news-mcp-wrapper-sports (AgenticIdentityToken), role News.Read.Sports


In [40]:
time.sleep(30)  # let the new role assignments reach Entra before the first token is issued
made = 0  # tool calls that reached the wrapper
for name, category in (('news-all-agent', 'business'), ('news-sports-agent', 'sports'), ('news-sports-agent', 'business')):
    response = checks.ask(project.get_openai_client(agent_name=name), checks.prompt(category))
    roles = [FALLBACK[name][0]]
    made += len(checks.called_tools(response))
    print(f'{name:<18} {category:<9} {checks.classify(response, roles, category):<11} tools called: {[t for t, _ in checks.called_tools(response)]}')
time.sleep(10)

print('\nWrapper log:')
client_ids = {name: agent_identity(name)['client_id'] for name in FALLBACK}
for e in wrapper_events('tools/call')[-made:]:
    name = fallback_ids[e['oid']]
    print(f"  idtyp={e['idtyp']}  oid={name}  azp is its client id={e.get('azp') == client_ids[name]}  "
          f"roles={e['roles']}  {e['tool']}  {e['decision']}")

news-all-agent     business  articles    tools called: ['get_business_headlines']
news-sports-agent  sports    articles    tools called: ['get_sports_headlines']
news-sports-agent  business  hidden      tools called: []

Wrapper log:
  idtyp=app  oid=news-all-agent  azp is its client id=True  roles=['News.Read.All']  get_business_headlines  allowed
  idtyp=app  oid=news-sports-agent  azp is its client id=True  roles=['News.Read.Sports']  get_sports_headlines  allowed


**Observed on 2026-10-30.** On the first run `news-sports-agent`, asked for business, called `get_business_headlines` although the wrapper had listed only `get_sports_headlines`: the model guessed the name from the one it had. Foundry checked the call against the server's list and failed the whole request with HTTP 400 `tool_user_error`, "Tool get_business_headlines not found on remote server". Nothing reached the wrapper. `checks.ask` turns that error into a `hidden` outcome, so the matrix records it instead of stopping.

## Step 10: Revoke user B's `News.Read.Science`

Deleting the assignment changes what Entra puts in user B's next token for `news-mcp-{suffix}`. A token Foundry obtained and cached before the change still carries `News.Read.Science` until it expires. 08-05c-03 measures both sides: a call through its connection, which has never fetched a token for user B, and a repeat call through this track. Run it soon; the time of the revocation goes to `.env` as `NEWS_SCIENCE_REVOKED_AT`.

In [ ]:
science = authz.permission_id(SUFFIX, 'News.Read.Science')
for a in graph('GET', assigned_to)['value']:
    if a['principalId'] == TEST_USER_ID and a['appRoleId'] == science:
        graph('DELETE', f"{assigned_to}/{a['id']}")
revoked_at = datetime.now(timezone.utc).isoformat(timespec='seconds')
env_set(NEWS_SCIENCE_REVOKED_AT=revoked_at)
roles_b = sorted(r for r, i in ((r, authz.permission_id(SUFFIX, r)) for r in authz.ROLES)
                 if i in {a['appRoleId'] for a in graph('GET', assigned_to)['value'] if a['principalId'] == TEST_USER_ID})
print(f'Revoked News.Read.Science from user B at {revoked_at}')
print(f'User B now holds: {roles_b}')

## Cleanup

Leave everything in place for 08-05c-03. The full teardown is the Cleanup section of [08-05c-01](08-05c-01-identity-and-agent.ipynb), which removes this notebook's resources too. To remove only Track A, uncomment the cell.

In [ ]:
# for name in ('news-all-agent', 'news-sports-agent'):
#     print(name, call('DELETE', f'{PROJECT_ENDPOINT}/agents/{name}?api-version=v1&force=true', FOUNDRY).status_code)
# for name in ('news-mcp-wrapper', 'news-mcp-wrapper-all', 'news-mcp-wrapper-sports'):
#     print(name, call('DELETE', ident.connection_url(SUBSCRIPTION_ID, SUFFIX, name, CONNECTION_API), ARM).status_code)
# subprocess.run(['az', 'group', 'delete', '-n', RG, '--yes'], check=True)